In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv('./data/synthetic_learning_data.csv')

In [3]:
df

,student_id,course_id,chapter_order,time_spent,score,completion_status
0,S001,C3,1,53.2,60.8,1
1,S001,C3,2,27.0,100.0,1
2,S001,C3,3,43.6,56.1,1
3,S001,C3,4,31.7,63.8,1
4,S001,C3,5,37.8,72.5,1
...,...,...,...,...,...,...
955,S120,C3,4,54.3,66.9,0
956,S120,C3,5,37.7,77.0,0
957,S120,C3,6,37.9,74.0,0
958,S120,C3,7,36.6,48.1,0


In [5]:
df.head()

,student_id,course_id,chapter_order,time_spent,score,completion_status
0,S001,C3,1,53.2,60.8,1
1,S001,C3,2,27.0,100.0,1
2,S001,C3,3,43.6,56.1,1
3,S001,C3,4,31.7,63.8,1
4,S001,C3,5,37.8,72.5,1


In [6]:
df.columns

Index(['student_id', 'course_id', 'chapter_order', 'time_spent', 'score',
       'completion_status'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   student_id         960 non-null    object 
 1   course_id          960 non-null    object 
 2   chapter_order      960 non-null    int64  
 3   time_spent         960 non-null    float64
 4   score              960 non-null    float64
 5   completion_status  960 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 45.1+ KB


In [8]:
def create_student_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert chapter-level data into student-level features
    """

    student_df = (
        df.groupby(["student_id", "course_id"])
        .agg(
            avg_time_spent=("time_spent", "mean"),
            max_time_spent=("time_spent", "max"),
            avg_score=("score", "mean"),
            min_score=("score", "min"),
            chapters_completed=("chapter_order", "count"),
            completion_status=("completion_status", "max")
        )
        .reset_index()
    )

    # Early performance (first 3 chapters)
    early_df = (
        df[df["chapter_order"] <= 3]
        .groupby(["student_id", "course_id"])
        .agg(early_avg_score=("score", "mean"))
        .reset_index()
    )

    student_df = student_df.merge(
        early_df,
        on=["student_id", "course_id"],
        how="left"
    )

    return student_df


In [12]:
student_df = create_student_features(df)

In [9]:
TARGET_COL = "completion_status"

categorical_cols = ["course_id"]

numerical_cols = [
    "avg_time_spent",
    "max_time_spent",
    "avg_score",
    "min_score",
    "chapters_completed",
    "early_avg_score"
]


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num_pipeline", num_pipeline, numerical_cols),
        ("cat_pipeline", cat_pipeline, categorical_cols)
    ]
)


In [13]:
from sklearn.model_selection import train_test_split

X = student_df.drop(columns=["student_id", TARGET_COL])
y = student_df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


In [14]:
X = student_df.drop(columns=["student_id", TARGET_COL])
y = student_df[TARGET_COL]


In [16]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)


In [17]:
from sklearn.pipeline import Pipeline

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

clf.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['avg_time_spent',
                                                   'max_time_spent',
                                                   'avg_score', 'min_score',
                                                   'chapters_completed',
                                                   'early_avg_score']),
                                                 ('cat_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['course_id'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

print("Training Metrics:")
print({
    "accuracy": accuracy_score(y_train, y_train_pred),
    "precision": precision_score(y_train, y_train_pred),
    "recall": recall_score(y_train, y_train_pred),
    "f1": f1_score(y_train, y_train_pred)
})

print("\nTesting Metrics:")
print({
    "accuracy": accuracy_score(y_test, y_test_pred),
    "precision": precision_score(y_test, y_test_pred),
    "recall": recall_score(y_test, y_test_pred),
    "f1": f1_score(y_test, y_test_pred)
})


Training Metrics:
{'accuracy': 0.6547619047619048, 'precision': 0.8095238095238095, 'recall': 0.6181818181818182, 'f1': 0.7010309278350516}

Testing Metrics:
{'accuracy': 0.4166666666666667, 'precision': 0.5555555555555556, 'recall': 0.43478260869565216, 'f1': 0.4878048780487805}


In [20]:
y_prob = clf.predict_proba(X_test)[:, 1]

risk_threshold = 0.4
y_risk_pred = (y_prob > risk_threshold).astype(int)

print("\nRisk Detection Metrics (Threshold = 0.4):")
print({
    "accuracy": accuracy_score(y_test, y_risk_pred),
    "precision": precision_score(y_test, y_risk_pred),
    "recall": recall_score(y_test, y_risk_pred),
    "f1": f1_score(y_test, y_risk_pred)
})



Risk Detection Metrics (Threshold = 0.4):
{'accuracy': 0.5277777777777778, 'precision': 0.6, 'recall': 0.782608695652174, 'f1': 0.6792452830188679}
